In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSVs into DataFrames (keep in memory for follow-ups)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# Helper: convert integer-second timestamp to timezone-aware datetime in UTC+8 (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
def to_shanghai_ts(s):
    return pd.to_datetime(s, unit='s', utc=True).dt.tz_convert(tz)

# Prepare overall summary info for each file: total rows, min and max timestamp (converted)
metric_overall = pd.DataFrame([{
    'total_rows': len(metric_df),
    'min_timestamp': to_shanghai_ts(metric_df['timestamp']).min() if len(metric_df) else pd.NaT,
    'max_timestamp': to_shanghai_ts(metric_df['timestamp']).max() if len(metric_df) else pd.NaT
}])

trace_overall = pd.DataFrame([{
    'total_rows': len(trace_df),
    'min_timestamp': to_shanghai_ts(trace_df['timestamp']).min() if len(trace_df) else pd.NaT,
    'max_timestamp': to_shanghai_ts(trace_df['timestamp']).max() if len(trace_df) else pd.NaT
}])

log_overall = pd.DataFrame([{
    'total_rows': len(log_df),
    'min_timestamp': to_shanghai_ts(log_df['timestamp']).min() if len(log_df) else pd.NaT,
    'max_timestamp': to_shanghai_ts(log_df['timestamp']).max() if len(log_df) else pd.NaT
}])

error_overall = pd.DataFrame([{
    'total_rows': len(error_logs_df),
    'min_timestamp': to_shanghai_ts(error_logs_df['timestamp']).min() if len(error_logs_df) else pd.NaT,
    'max_timestamp': to_shanghai_ts(error_logs_df['timestamp']).max() if len(error_logs_df) else pd.NaT
}])

# 1) metric.csv grouping by (cmdb_id, kpi_name): count and global P95 of 'value'
if 'value' in metric_df.columns and 'cmdb_id' in metric_df.columns and 'kpi_name' in metric_df.columns:
    metric_groups = metric_df.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value'].agg(
        count='count',
        p95_value=lambda x: x.quantile(0.95)
    ).reset_index()
    metric_groups['p95_value'] = metric_groups['p95_value'].round(6)
    metric_groups_top20 = metric_groups.sort_values('count', ascending=False).head(20).reset_index(drop=True)
else:
    metric_groups_top20 = pd.DataFrame(columns=['cmdb_id','kpi_name','count','p95_value'])

# 2) trace.csv grouping by (cmdb_id, trace_name)
if 'value' in trace_df.columns and 'cmdb_id' in trace_df.columns and 'trace_name' in trace_df.columns:
    trace_groups = trace_df.groupby(['cmdb_id', 'trace_name'], dropna=False)['value'].agg(
        count='count',
        p95_value=lambda x: x.quantile(0.95)
    ).reset_index()
    trace_groups['p95_value'] = trace_groups['p95_value'].round(6)
    trace_groups_top20 = trace_groups.sort_values('count', ascending=False).head(20).reset_index(drop=True)
else:
    trace_groups_top20 = pd.DataFrame(columns=['cmdb_id','trace_name','count','p95_value'])

# 3) log.csv grouping by (cmdb_id, log_name)
if 'value' in log_df.columns and 'cmdb_id' in log_df.columns and 'log_name' in log_df.columns:
    log_groups = log_df.groupby(['cmdb_id', 'log_name'], dropna=False)['value'].agg(
        count='count',
        p95_value=lambda x: x.quantile(0.95)
    ).reset_index()
    log_groups['p95_value'] = log_groups['p95_value'].round(6)
    log_groups_top20 = log_groups.sort_values('count', ascending=False).head(20).reset_index(drop=True)
else:
    log_groups_top20 = pd.DataFrame(columns=['cmdb_id','log_name','count','p95_value'])

# 4) error_logs.csv: up to 5 sample rows with converted timestamps
if not error_logs_df.empty:
    error_samples = error_logs_df[['timestamp','cmdb_id','message']].copy()
    error_samples['timestamp'] = to_shanghai_ts(error_samples['timestamp'])
    error_samples = error_samples.head(5).reset_index(drop=True)
else:
    error_samples = error_logs_df[['timestamp','cmdb_id','message']].head(0)

# Final displayed variables (compact)
metric_overall, metric_groups_top20, trace_overall, trace_groups_top20, log_overall, log_groups_top20, error_overall, error_samples

```
Out[1]:
```
```python
# Compose a plain-English summary using the previously loaded summary variables
# (metric_overall, metric_groups_top20, trace_overall, trace_groups_top20,
#  log_overall, log_groups_top20, error_overall, error_samples)

def fmt_ts(x):
    if pd.isna(x):
        return "N/A"
    return str(x)

lines = []

# Metric summary
mo = metric_overall.iloc[0]
lines.append(f"metric.csv: total_rows={int(mo['total_rows'])}, time_range={fmt_ts(mo['min_timestamp'])} to {fmt_ts(mo['max_timestamp'])}.")
if not metric_groups_top20.empty:
    lines.append("Top metric groups (up to 3 examples from top 20 sorted by count):")
    for _, r in metric_groups_top20.head(3).iterrows():
        lines.append(f" - {r['cmdb_id']} / {r['kpi_name']}: count={int(r['count'])}, p95_value={float(r['p95_value']):.6g}")
    lines.append(f"Displayed top {len(metric_groups_top20)} groups (sorted by count).")
else:
    lines.append("No metric groups available.")

# Trace summary
tr = trace_overall.iloc[0]
lines.append(f"\ntrace.csv: total_rows={int(tr['total_rows'])}, time_range={fmt_ts(tr['min_timestamp'])} to {fmt_ts(tr['max_timestamp'])}.")
if not trace_groups_top20.empty:
    lines.append("Top trace groups (up to 3 examples):")
    for _, r in trace_groups_top20.head(3).iterrows():
        lines.append(f" - {r['cmdb_id']} / {r['trace_name']}: count={int(r['count'])}, p95_value={float(r['p95_value']):.6g}")
    lines.append(f"Displayed top {len(trace_groups_top20)} groups (sorted by count).")
else:
    lines.append("No trace groups available.")

# Log summary
lo = log_overall.iloc[0]
lines.append(f"\nlog.csv: total_rows={int(lo['total_rows'])}, time_range={fmt_ts(lo['min_timestamp'])} to {fmt_ts(lo['max_timestamp'])}.")
if not log_groups_top20.empty:
    lines.append("Top log groups (up to 3 examples):")
    for _, r in log_groups_top20.head(3).iterrows():
        lines.append(f" - {r['cmdb_id']} / {r['log_name']}: count={int(r['count'])}, p95_value={float(r['p95_value']):.6g}")
    lines.append(f"Displayed top {len(log_groups_top20)} groups (sorted by count).")
else:
    lines.append("No log groups available.")

# Error logs summary
eo = error_overall.iloc[0]
lines.append(f"\nerror_logs.csv: total_rows={int(eo['total_rows'])}.")
if int(eo['total_rows']) == 0:
    lines.append("No error log entries found (no samples).")
else:
    sample_n = len(error_samples)
    lines.append(f"Provided up to {sample_n} sample rows from error_logs.csv.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows             min_timestamp             max_timestamp
0       26775 2021-03-07 03:30:00+08:00 2021-03-07 04:00:00+08:00,      cmdb_id                                           kpi_name  count     p95_value
0   apache02                             OSLinux-CPU_CPU_CPUWio     31  1.266770e-01
1       IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31  7.500000e+01
2   apache02                        OSLinux-CPU_CPU_CPUUserTime     31  1.598500e-01
3       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31  5.796209e+07
4   apache02                         OSLinux-CPU_CPU_CPUSysTime     31  3.821420e-01
5   apache02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31  0.000000e+00
6   apache02                            OSLinux-CPU_CPU_CPULoad     31  1.300000e-01
7   apache02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31  0.000000e+00
8   Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00
9   Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...     31  4.800000e+01
10  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     31  3.614060e+05
11  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31  2.841973e+08
12  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...     31  5.360265e+05
13  apache01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31  0.000000e+00
14  Tomcat04    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter     31  0.000000e+00
15  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...     31  0.000000e+00
16  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...     31  0.000000e+00
17  Tomcat04        Tomcat-Sessions_7441--_SESSIONActiveCounter     31  0.000000e+00
18  Tomcat04     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter     31  0.000000e+00
19  Tomcat04     Tomcat-Sessions_7441--_SESSIONRejectedSessions     31  0.000000e+00,    total_rows             min_timestamp             max_timestamp
0        2823 2021-03-07 03:30:00+08:00 2021-03-07 03:59:00+08:00,      cmdb_id                         trace_name  count  p95_value
0       IG01           trace.self.duration_mean     25   0.059490
1       IG01            trace.self.duration_p95     25   0.099900
2       IG01               trace.self.row_count     25  29.200000
3       MG01           trace.self.duration_mean     23   0.040971
4       MG01               trace.self.row_count     23  19.700000
5       MG01            trace.self.duration_p95     23   0.091360
6       IG02           trace.self.duration_mean     22   0.074498
7       IG02            trace.self.duration_p95     22   0.339650
8       IG02               trace.self.row_count     22  29.900000
9   Tomcat04               trace.self.row_count     22  47.800000
10  Tomcat04            trace.self.duration_p95     22   0.063225
11  Tomcat04           trace.self.duration_mean     22   0.009281
12      MG02            trace.self.duration_p95     21   0.141400
13      MG02               trace.self.row_count     21  17.000000
14      MG02           trace.self.duration_mean     21   0.039615
15  Tomcat01            trace.self.duration_p95     20   0.049038
16  Tomcat01           trace.self.duration_mean     20   0.010479
17  Tomcat01               trace.self.row_count     20  57.450000
18      MG01  trace.from_Tomcat04.duration_mean     19   0.053925
19      MG01      trace.from_Tomcat04.row_count     19   6.600000,    total_rows             min_timestamp             max_timestamp
0         254 2021-03-07 03:30:00+08:00 2021-03-07 04:00:00+08:00,      cmdb_id          log_name  count  p95_value
0   apache02     log.row_count     22    14.9500
1   apache02   log.error_count     22     0.0000
2   Tomcat04     log.row_count     22     8.0000
3   Tomcat04   log.error_count     22     0.0000
4   Tomcat01   log.error_count     21     0.0000
5   Tomcat01     log.row_count     21     8.0000
6   apache01   log.error_count     20     0.0000
7   apache01     log.row_count     20    16.1500
8   Tomcat03     log.row_count     19     7.9000
9   Tomcat03   log.error_count     19     0.0000
10  Tomcat02   log.error_count     18     0.0000
11  Tomcat02     log.row_count     18    29.8000
12  Tomcat02  log.gc_pause_max      4     0.1355
13  Tomcat01  log.gc_pause_max      2     0.0590
14  Tomcat03  log.gc_pause_max      2     0.0680
15  Tomcat04  log.gc_pause_max      2     0.0695,    total_rows min_timestamp max_timestamp
0           0           NaT           NaT, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse metric_df from kernel
tz = pytz.timezone('Asia/Shanghai')

# 1) Global per-group P95 and total count across entire metric.csv
metric_global = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(total_count='count', p95_value=lambda x: x.quantile(0.95))
    .reset_index()
)
metric_global['p95_value'] = metric_global['p95_value'].round(6)

# 2) Prepare timestamp in UTC+8 and merge global thresholds into metric rows
metric_with_p95 = metric_df.merge(metric_global, on=['cmdb_id', 'kpi_name'], how='left')
metric_with_p95['ts_sh'] = pd.to_datetime(metric_with_p95['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Define incident window in UTC+8 (inclusive)
window_start = pd.Timestamp('2021-03-07 03:30:00', tz=tz)
window_end   = pd.Timestamp('2021-03-07 04:00:00', tz=tz)

# Filter rows within the window
metric_window = metric_with_p95[(metric_with_p95['ts_sh'] >= window_start) & (metric_with_p95['ts_sh'] <= window_end)].copy()

# Mark anomalies: value >= global P95 (threshold computed from full series)
metric_window['is_anomaly'] = metric_window['value'] >= metric_window['p95_value']

# Compute per-group within-window stats
window_counts = metric_window.groupby(['cmdb_id','kpi_name'], dropna=False)['value'].count().reset_index().rename(columns={'value':'window_count'})
max_in_window = metric_window.groupby(['cmdb_id','kpi_name'], dropna=False)['value'].max().reset_index().rename(columns={'value':'max_value_in_window'})

anomalies_df = metric_window[metric_window['is_anomaly']].copy()
anomaly_counts = anomalies_df.groupby(['cmdb_id','kpi_name'], dropna=False)['value'].count().reset_index().rename(columns={'value':'anomaly_count'})
earliest_anomaly = anomalies_df.groupby(['cmdb_id','kpi_name'], dropna=False)['ts_sh'].min().reset_index().rename(columns={'ts_sh':'earliest_anomaly_timestamp'})

# 3) Merge everything into a final summary, keep only groups with anomaly_count > 0
final_df = metric_global.merge(window_counts, on=['cmdb_id','kpi_name'], how='left') \
                        .merge(anomaly_counts, on=['cmdb_id','kpi_name'], how='left') \
                        .merge(earliest_anomaly, on=['cmdb_id','kpi_name'], how='left') \
                        .merge(max_in_window, on=['cmdb_id','kpi_name'], how='left')

# Fill NA numeric window/anomaly counts with 0
final_df['window_count'] = final_df['window_count'].fillna(0).astype(int)
final_df['anomaly_count'] = final_df['anomaly_count'].fillna(0).astype(int)

# Filter groups with anomalies and sort
anomalous_groups = final_df[final_df['anomaly_count'] > 0].copy()
anomalous_groups = anomalous_groups.sort_values(['anomaly_count','window_count'], ascending=[False, False])

# Compact output: select and order columns, limit to 20 rows
anomalous_compact = anomalous_groups[
    ['cmdb_id','kpi_name','total_count','p95_value','window_count','anomaly_count','earliest_anomaly_timestamp','max_value_in_window']
].head(20).reset_index(drop=True)

# Counts
total_groups_scanned = metric_global.shape[0]
groups_with_anomalies = anomalous_groups.shape[0]

# Display compact results (variables kept in kernel for follow-ups)
anomalous_compact, total_groups_scanned, groups_with_anomalies

```
Out[2]:
```
Summary of results (computed with global P95 thresholds from the full metric.csv):

- Total metric groups scanned: 1701. Groups with at least one anomaly in the incident window: 1693.
- The compact table (top 20 groups) shows many groups with anomaly_count = 31 and window_count = 31, i.e., every sample in the window (2021-03-07 03:30 to 2021-03-07 04:00, Asia/Shanghai UTC+8) exceeded the global P95 threshold. The earliest anomaly for these groups is 2021-03-07 03:30:00+08:00.
- Examples from the top results: IG01 and IG02 have JVM-related metrics flagged (JVM_Memory HeapMemoryMax and NoHeapMemoryUsed), MG01/MG02 show JVM HeapMemoryMax and NoHeapMemoryUsed, and Mysql01/Mysql02 show several OSLinux network/process KPIs (e.g., TCP-CLOSE-WAIT, TCP-FIN-WAIT, PROCESS_PROCNoZombies) all with anomaly_count = 31 in the window.
- In short: anomalies are widespread across hosts and KPI types during the entire incident window, with many JVM memory and process/network KPIs continuously above their global P95 thresholds.

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                           kpi_name  total_count     p95_value  window_count  anomaly_count earliest_anomaly_timestamp  max_value_in_window
0      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax           31  2.040136e+09            31             31  2021-03-07 03:30:00+08:00         2.040136e+09
1      IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed           31  5.796209e+07            31             31  2021-03-07 03:30:00+08:00         5.796209e+07
2      IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies           31  1.000000e+00            31             31  2021-03-07 03:30:00+08:00         1.000000e+00
3      IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...           31  3.000000e-01            31             31  2021-03-07 03:30:00+08:00         3.000000e-01
4      IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount           31  3.000000e+00            31             31  2021-03-07 03:30:00+08:00         3.000000e+00
5      IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax           31  2.040136e+09            31             31  2021-03-07 03:30:00+08:00         2.040136e+09
6      IG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies           31  1.000000e+00            31             31  2021-03-07 03:30:00+08:00         1.000000e+00
7      IG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...           31  3.000000e-01            31             31  2021-03-07 03:30:00+08:00         3.000000e-01
8      IG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount           31  3.000000e+00            31             31  2021-03-07 03:30:00+08:00         3.000000e+00
9      MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax           31  2.040136e+09            31             31  2021-03-07 03:30:00+08:00         2.040136e+09
10     MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed           31  5.068773e+08            31             31  2021-03-07 03:30:00+08:00         5.068773e+08
11     MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax           31  2.040136e+09            31             31  2021-03-07 03:30:00+08:00         2.040136e+09
12     MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed           31  9.473127e+08            31             31  2021-03-07 03:30:00+08:00         9.473127e+08
13  Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT           31  0.000000e+00            31             31  2021-03-07 03:30:00+08:00         0.000000e+00
14  Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT           31  0.000000e+00            31             31  2021-03-07 03:30:00+08:00         0.000000e+00
15  Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies           31  0.000000e+00            31             31  2021-03-07 03:30:00+08:00         0.000000e+00
16  Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount           31  7.000000e+00            31             31  2021-03-07 03:30:00+08:00         7.000000e+00
17  Mysql02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT           31  0.000000e+00            31             31  2021-03-07 03:30:00+08:00         0.000000e+00
18  Mysql02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT           31  0.000000e+00            31             31  2021-03-07 03:30:00+08:00         0.000000e+00
19  Mysql02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies           31  0.000000e+00            31             31  2021-03-07 03:30:00+08:00         0.000000e+00, 1701, 1693)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
import numpy as np

# Reuse trace_df loaded in kernel
tz = pytz.timezone('Asia/Shanghai')

# 1) Global per-group P95, P5, and total_count across full trace.csv
trace_global = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(total_count='count',
         p95_value=lambda x: x.quantile(0.95),
         p5_value = lambda x: x.quantile(0.05))
    .reset_index()
)

# Round thresholds for compactness but keep full precision in calculations
trace_global['p95_value'] = trace_global['p95_value'].astype(float)
trace_global['p5_value'] = trace_global['p5_value'].astype(float)

# 2) Filter trace.csv to the incident window (UTC+8)
trace_with_thresholds = trace_df.merge(trace_global, on=['cmdb_id','trace_name'], how='left')
trace_with_thresholds['ts_sh'] = pd.to_datetime(trace_with_thresholds['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

window_start = pd.Timestamp('2021-03-07 03:30:00', tz=tz)
window_end   = pd.Timestamp('2021-03-07 04:00:00', tz=tz)

trace_window = trace_with_thresholds[(trace_with_thresholds['ts_sh'] >= window_start) & (trace_with_thresholds['ts_sh'] <= window_end)].copy()

# Compute window stats per group
window_stats = trace_window.groupby(['cmdb_id','trace_name'], dropna=False)['value'].agg(
    window_count='count',
    max_value_in_window='max',
    min_value_in_window='min'
).reset_index()

# 3) Define anomaly types per group
def classify_trace_type(name):
    n = str(name).lower()
    if 'row_count' in n:
        return 'low_traffic'
    if 'duration' in n or 'duration_mean' in n or 'duration_p95' in n:
        return 'high_latency'
    return None

trace_global['anomaly_type'] = trace_global['trace_name'].apply(classify_trace_type)

# Merge window stats and thresholds
trace_merged = trace_global.merge(window_stats, on=['cmdb_id','trace_name'], how='left')

# Prepare anomalies within window
trace_window['anomaly_type'] = trace_window['trace_name'].apply(classify_trace_type)

# High latency anomalies: value >= p95
hl = trace_window[trace_window['anomaly_type']=='high_latency'].copy()
hl['is_anomaly'] = hl['value'] >= hl['p95_value']

# Low traffic anomalies: value <= p5
lt = trace_window[trace_window['anomaly_type']=='low_traffic'].copy()
lt['is_anomaly'] = lt['value'] <= lt['p5_value']

# Aggregate anomaly counts and earliest timestamps
hl_counts = hl[hl['is_anomaly']].groupby(['cmdb_id','trace_name'], dropna=False).agg(
    anomaly_count=('is_anomaly','sum'),
    earliest_anomaly_timestamp=('ts_sh','min')
).reset_index()

lt_counts = lt[lt['is_anomaly']].groupby(['cmdb_id','trace_name'], dropna=False).agg(
    anomaly_count=('is_anomaly','sum'),
    earliest_anomaly_timestamp=('ts_sh','min')
).reset_index()

anomaly_counts = pd.concat([hl_counts, lt_counts], ignore_index=True) if (not hl_counts.empty or not lt_counts.empty) else pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp'])

# Merge anomaly info back
final_trace = trace_merged.merge(anomaly_counts, on=['cmdb_id','trace_name'], how='left')

# Fill NaNs for counts
final_trace['window_count'] = final_trace['window_count'].fillna(0).astype(int)
final_trace['anomaly_count'] = final_trace['anomaly_count'].fillna(0).astype(int)

# Ensure min/max values present (may be NaN if no rows in window)
# Keep as floats; fill NaN for calculations carefully
# 4) Compute severity:
# - high_latency: severity = max_value_in_window / p95_value
# - low_traffic: severity = p5_value / min_value_in_window
def compute_severity(row):
    atype = row['anomaly_type']
    if row['anomaly_count'] <= 0:
        return 0.0
    if atype == 'high_latency':
        if pd.isna(row['max_value_in_window']) or row['p95_value'] == 0:
            return np.inf if (not pd.isna(row['max_value_in_window']) and row['max_value_in_window']>0) else 0.0
        return row['max_value_in_window'] / row['p95_value']
    if atype == 'low_traffic':
        if pd.isna(row['min_value_in_window']) or row['min_value_in_window'] == 0:
            return np.inf
        return row['p5_value'] / row['min_value_in_window']
    return 0.0

final_trace['severity'] = final_trace.apply(compute_severity, axis=1)

# Keep only groups with anomaly_count > 0
anomalous_traces = final_trace[final_trace['anomaly_count'] > 0].copy()

# Prepare compact output columns and sort by severity descending, then anomaly_count desc
anomalous_compact = anomalous_traces[
    ['cmdb_id','trace_name','total_count','p95_value','p5_value','window_count','anomaly_type','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','min_value_in_window','severity']
].sort_values(['severity','anomaly_count'], ascending=[False, False]).head(20).reset_index(drop=True)

# Round numeric columns for compact display
anomalous_compact['p95_value'] = anomalous_compact['p95_value'].round(6)
anomalous_compact['p5_value'] = anomalous_compact['p5_value'].round(6)
anomalous_compact['max_value_in_window'] = anomalous_compact['max_value_in_window'].astype(float).round(6)
anomalous_compact['min_value_in_window'] = anomalous_compact['min_value_in_window'].astype(float).round(6)
anomalous_compact['severity'] = anomalous_compact['severity'].replace([np.inf, -np.inf], np.nan).round(6)

# Two counts
total_trace_groups_scanned = trace_global.shape[0]
trace_groups_with_anomalies = anomalous_traces.shape[0]

# Display compact results (cached full intermediates: trace_global, trace_with_thresholds, trace_window, final_trace)
anomalous_compact, total_trace_groups_scanned, trace_groups_with_anomalies

```
Out[3]:
```
Summary of trace anomaly analysis (using global P95/P5 from full trace.csv, window 2021-03-07 03:30–04:00 Asia/Shanghai):

- Total trace groups scanned: 228.  
- Trace groups with anomalies in the window: 228 (every scanned group showed at least one anomaly under the defined rules).

Top findings (compact top 20 by severity):
- Highest severity items (example top entries):
  - MG01 / trace.from_Tomcat01.duration_p95 — high_latency — anomaly_count=1 — earliest 2021-03-07 03:42:00+08:00 — severity ≈ 3.30 (max / p95).
  - Tomcat01 / trace.to_MG01.duration_p95 — high_latency — anomaly_count=1 — earliest 2021-03-07 03:42:00+08:00 — severity ≈ 3.30.
  - MG01 / trace.to_dockerB1.duration_p95 and dockerB1 / trace.from_MG01.duration_p95 — high_latency — severity ≈ 2.71.
  - MG02 / trace.self.duration_p95 — high_latency — anomaly_count=2 — earliest 2021-03-07 03:41:00+08:00 — severity ≈ 2.66.
  - dockerB1 / trace.self.row_count — low_traffic — anomaly_count=1 — earliest 2021-03-07 03:32:00+08:00 — severity = p5 / min ≈ 2.50.
  - MG01 / trace.self.row_count — low_traffic — anomaly_count=4 — earliest 2021-03-07 03:32:00+08:00 — severity = 2.00.

Patterns and interpretation:
- Most anomalies are high_latency on duration-type traces (duration, duration_mean, duration_p95): values exceeded their global P95 thresholds in the window. These show severity ratios typically ~2.0–3.3 (max_value_in_window / p95_value).
- Some low_traffic anomalies on row_count traces (value <= global P5) were observed (e.g., dockerB1 and MG01 row_count entries).
- Earliest anomaly times for many high-severity duration anomalies cluster around 03:41–03:42+08:00; row_count low-traffic anomalies appear earlier (e.g., ~03:32+08:00).
- Anomaly counts per group are generally small (1–2) but some groups (e.g., MG01 trace.self.row_count) had multiple low-traffic occurrences (anomaly_count=4).

Conclusion: anomalies are widespread across services and trace types during the incident window, dominated by elevated trace durations (high_latency) with notable low-traffic hits on some row_count traces.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                        trace_name  total_count   p95_value  p5_value  window_count  anomaly_type  anomaly_count earliest_anomaly_timestamp  max_value_in_window  min_value_in_window  severity
0       MG01  trace.from_Tomcat01.duration_p95           17    0.103400  0.007800            17  high_latency              1  2021-03-07 03:42:00+08:00             0.341000             0.007000  3.297872
1   Tomcat01        trace.to_MG01.duration_p95           17    0.103400  0.007800            17  high_latency              1  2021-03-07 03:42:00+08:00             0.341000             0.007000  3.297872
2       MG01    trace.to_dockerB1.duration_p95           14    0.140947  0.004585            14  high_latency              1  2021-03-07 03:42:00+08:00             0.382650             0.004000  2.714841
3   dockerB1      trace.from_MG01.duration_p95           14    0.140947  0.004585            14  high_latency              1  2021-03-07 03:42:00+08:00             0.382650             0.004000  2.714841
4       MG02           trace.self.duration_p95           21    0.141400  0.007000            21  high_latency              2  2021-03-07 03:41:00+08:00             0.376800             0.006000  2.664781
5       MG01   trace.to_dockerB1.duration_mean           14    0.051746  0.004216            14  high_latency              1  2021-03-07 03:42:00+08:00             0.131750             0.004000  2.546088
6   dockerB1     trace.from_MG01.duration_mean           14    0.051746  0.004216            14  high_latency              1  2021-03-07 03:42:00+08:00             0.131750             0.004000  2.546088
7   dockerB1              trace.self.row_count           16  101.000000  2.500000            16   low_traffic              1  2021-03-07 03:32:00+08:00           221.000000             1.000000  2.500000
8       MG01           trace.self.duration_p95           23    0.091360  0.009975            23  high_latency              2  2021-03-07 03:39:00+08:00             0.224500             0.009000  2.457312
9   dockerB1           trace.self.duration_p95           16    0.018138  0.001000            16  high_latency              1  2021-03-07 03:41:00+08:00             0.039550             0.001000  2.180565
10      IG01           trace.self.duration_p95           25    0.099900  0.016400            25  high_latency              2  2021-03-07 03:43:00+08:00             0.211150             0.014000  2.113614
11  dockerB1          trace.self.duration_mean           16    0.007383  0.000879            16  high_latency              1  2021-03-07 03:41:00+08:00             0.015237             0.000667  2.063795
12      MG02  trace.from_Tomcat01.duration_p95           13    0.257230  0.007600            13  high_latency              1  2021-03-07 03:41:00+08:00             0.524800             0.007000  2.040197
13  Tomcat01        trace.to_MG02.duration_p95           13    0.257230  0.007600            13  high_latency              1  2021-03-07 03:41:00+08:00             0.524800             0.007000  2.040197
14      MG01              trace.self.row_count           23   19.700000  2.000000            23   low_traffic              4  2021-03-07 03:32:00+08:00            71.000000             1.000000  2.000000
15      MG02    trace.to_dockerB2.duration_p95           13    0.028980  0.003600            13  high_latency              1  2021-03-07 03:42:00+08:00             0.056400             0.003000  1.946170
16  dockerB2      trace.from_MG02.duration_p95           13    0.028980  0.003600            13  high_latency              1  2021-03-07 03:42:00+08:00             0.056400             0.003000  1.946170
17      MG01  trace.from_Tomcat03.duration_p95           13    0.239560  0.007000            13  high_latency              1  2021-03-07 03:42:00+08:00             0.461350             0.007000  1.925822
18  Tomcat03        trace.to_MG01.duration_p95           13    0.239560  0.007000            13  high_latency              1  2021-03-07 03:42:00+08:00             0.461350             0.007000  1.925822
19      IG02    trace.to_Tomcat04.duration_p95           16    0.122812  0.013750            16  high_latency              1  2021-03-07 03:42:00+08:00             0.236250             0.013000  1.923664, 228, 228)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse cached variables from previous steps:
# metric_global, final_df (per-(cmdb_id,kpi_name) with p95_value, window_count, anomaly_count, earliest_anomaly_timestamp, max_value_in_window)

tz = pytz.timezone('Asia/Shanghai')

# Components of interest (restricted set)
components_interest = [
    "apache01","apache02",
    "Tomcat01","Tomcat02","Tomcat03","Tomcat04",
    "MG01","MG02","IG01","IG02",
    "Mysql01","Mysql02","Redis01","Redis02"
]

# 1) Compute total_kpi_groups_scanned per component (within the restricted set)
metric_global_subset = metric_global[metric_global['cmdb_id'].isin(components_interest)].copy()
total_kpi_groups_scanned = metric_global_subset.groupby('cmdb_id', dropna=False).size().rename('total_kpi_groups_scanned').reset_index()

# 2) Prepare per-kpi severity and filter to components of interest
df_kpi = final_df.copy()
df_kpi = df_kpi[df_kpi['cmdb_id'].isin(components_interest)].copy()

# Normalize earliest_anomaly_timestamp to timezone-aware Asia/Shanghai where possible
# Use to_datetime with errors='coerce' and utc=True then convert to Asia/Shanghai
df_kpi['earliest_anomaly_timestamp'] = pd.to_datetime(df_kpi['earliest_anomaly_timestamp'], errors='coerce', utc=True)
df_kpi['earliest_anomaly_timestamp'] = df_kpi['earliest_anomaly_timestamp'].dt.tz_convert(tz)

# Compute severity per KPI (safe division)
def compute_kpi_severity(row):
    p95 = row.get('p95_value', np.nan)
    maxv = row.get('max_value_in_window', np.nan)
    try:
        if pd.isna(p95) or float(p95) == 0.0:
            if not pd.isna(maxv) and float(maxv) > 0:
                return np.inf
            return 0.0
        if pd.isna(maxv):
            return 0.0
        return float(maxv) / float(p95)
    except Exception:
        return np.nan

df_kpi['severity'] = df_kpi.apply(compute_kpi_severity, axis=1)

# 3) Identify faulty KPI groups (anomaly_count > 0)
fault_kpis = df_kpi[df_kpi['anomaly_count'] > 0].copy()

# Per-component aggregations
comp_fault_counts = fault_kpis.groupby('cmdb_id', dropna=False).agg(
    faulty_kpi_count = ('kpi_name','count'),
    earliest_component_anomaly = ('earliest_anomaly_timestamp','min'),
    max_severity = ('severity','max')
).reset_index()

# Merge with total_kpi_groups_scanned
comp_summary = pd.merge(total_kpi_groups_scanned, comp_fault_counts, on='cmdb_id', how='left')

# Fill components with no faults with zeros/NaT
comp_summary['faulty_kpi_count'] = comp_summary['faulty_kpi_count'].fillna(0).astype(int)
comp_summary['earliest_component_anomaly'] = pd.to_datetime(comp_summary['earliest_component_anomaly'], errors='coerce', utc=True).dt.tz_convert(tz)
comp_summary['max_severity'] = comp_summary['max_severity'].replace([np.nan], 0.0)

# 4) For each component, list top 3 faulty KPI groups by severity
def top_kpis_for_component(df, comp, topn=3):
    sub = df[df['cmdb_id']==comp].copy()
    if sub.empty:
        return []
    sub = sub.sort_values('severity', ascending=False)
    rows = []
    for _, r in sub.head(topn).iterrows():
        k = r['kpi_name']
        sev = r['severity']
        ets = r.get('earliest_anomaly_timestamp', pd.NaT)
        sev_str = "inf" if np.isinf(sev) else f"{sev:.6f}"
        ets_str = str(ets) if not pd.isna(ets) else "N/A"
        rows.append(f"{k}|{sev_str}|{ets_str}")
    return rows

top_rows = []
for comp in comp_summary['cmdb_id']:
    tops = top_kpis_for_component(fault_kpis, comp, topn=3)
    while len(tops) < 3:
        tops.append("")
    top_rows.append(tops)

top_df = pd.DataFrame(top_rows, columns=['top_kpi_1','top_kpi_2','top_kpi_3'])
comp_summary = pd.concat([comp_summary, top_df], axis=1)

# Final compact table: components with faulty_kpi_count>0, sorted by max_severity desc, limit 20
comp_result = comp_summary[comp_summary['faulty_kpi_count'] > 0].copy()
comp_result = comp_result.sort_values('max_severity', ascending=False)

comp_result_compact = comp_result[
    ['cmdb_id','total_kpi_groups_scanned','faulty_kpi_count','max_severity','earliest_component_anomaly','top_kpi_1','top_kpi_2','top_kpi_3']
].copy()

# Round max_severity for display and replace inf with NaN to keep output compact
comp_result_compact['max_severity'] = comp_result_compact['max_severity'].replace([np.inf, -np.inf], np.nan).round(6)

comp_result_compact = comp_result_compact.head(20).reset_index(drop=True)

# Two counts
scanned_components_count = metric_global[metric_global['cmdb_id'].isin(components_interest)]['cmdb_id'].nunique()
components_with_faults_count = comp_result_compact.shape[0]

# Cache intermediates: metric_global_subset, df_kpi, fault_kpis, comp_summary
comp_result_compact, scanned_components_count, components_with_faults_count

```
Out[4]:
```
```python
summary = (
    "Component-level fault summary (restricted set of 14 components):\n"
    "- Components scanned: 14. Components with faulty KPI groups: 14 (every component had at least one faulty KPI).\n\n"
    "Top findings sorted by component max severity:\n"
    "1) Mysql02: total_kpi_groups_scanned=226, faulty_kpi_count=226, max_severity=20.00, earliest_anomaly=2021-03-07 03:30:00+08:00.\n"
    "   Top KPI examples: Mysql-MySQL_3306_Opened Tables (severity=20.00), Mysql-MySQL_3306_Table open cache misses (severity≈20.00), Mysql-MySQL_3306_Rows Read (severity≈3.83).\n\n"
    "2) Mysql01: total_kpi_groups_scanned=226, faulty_kpi_count=225, max_severity≈4.312, earliest_anomaly=2021-03-07 03:30:00+08:00.\n\n"
    "3) apache01: total_kpi_groups_scanned=81, faulty_kpi_count=81, max_severity≈3.333, earliest_anomaly=2021-03-07 03:30:00+08:00.\n\n"
    "4) MG01: total_kpi_groups_scanned=92, faulty_kpi_count=92, max_severity≈2.936, earliest_anomaly=2021-03-07 03:30:00+08:00.\n\n"
    "5) IG02: total_kpi_groups_scanned=90, faulty_kpi_count=89, max_severity≈2.857, earliest_anomaly=2021-03-07 03:30:00+08:00.\n\n"
    "6) Redis02: total_kpi_groups_scanned=109, faulty_kpi_count=109, max_severity≈2.333, earliest_anomaly=2021-03-07 03:30:00+08:00.\n\n"
    "7) apache02: total_kpi_groups_scanned=82, faulty_kpi_count=81, max_severity≈2.286, earliest_anomaly=2021-03-07 03:30:00+08:00.\n\n"
    "8) Tomcat02: total_kpi_groups_scanned=111, faulty_kpi_count=111, max_severity≈2.183, earliest_anomaly=2021-03-07 03:30:00+08:00.\n\n"
    "9) Redis01: total_kpi_groups_scanned=113, faulty_kpi_count=110, max_severity≈2.152, earliest_anomaly=2021-03-07 03:30:00+08:00.\n\n"
    "10) IG01: total_kpi_groups_scanned=90, faulty_kpi_count=88, max_severity≈2.016, earliest_anomaly=2021-03-07 03:30:00+08:00.\n\n"
    "11) Tomcat03: total_kpi_groups_scanned=109, faulty_kpi_count=109, max_severity=2.000, earliest_anomaly=2021-03-07 03:30:00+08:00.\n\n"
    "12) Tomcat04: total_kpi_groups_scanned=109, faulty_kpi_count=109, max_severity≈1.818, earliest_anomaly=2021-03-07 03:30:00+08:00.\n\n"
    "13) MG02: total_kpi_groups_scanned=92, faulty_kpi_count=92, max_severity≈1.818, earliest_anomaly=2021-03-07 03:30:00+08:00.\n\n"
    "14) Tomcat01: total_kpi_groups_scanned=108, faulty_kpi_count=108, max_severity≈1.667, earliest_anomaly=2021-03-07 03:30:00+08:00.\n\n"
    "Interpretation:\n"
    "- All 14 components had KPI groups exceeding their global P95 thresholds during the incident window; many components had nearly or fully saturated faults (e.g., Mysql02 and Mysql01 with ~226 KPI groups affected).\n"
    "- The most severe component is Mysql02 (severity 20x), followed by Mysql01 and apache01. Earliest anomalies for top components all start at the window start (2021-03-07 03:30+08:00).\n"
    "- Top faulty KPIs per component are dominated by disk, network, CPU, and MySQL-specific metrics (e.g., Opened Tables, disk throughput). These should be prioritized for root-cause investigation.\n\n"
    "Counts: scanned_components_count=14, components_with_faults_count=14."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id  total_kpi_groups_scanned  faulty_kpi_count  max_severity earliest_component_anomaly                                          top_kpi_1                                          top_kpi_2                                          top_kpi_3
0    Mysql02                       226               226     20.000000  2021-03-07 03:30:00+08:00  Mysql-MySQL_3306_Opened Tables|20.000000|2021-...  Mysql-MySQL_3306_Table open cache misses|20.00...  Mysql-MySQL_3306_Rows Read|3.832456|2021-03-07...
1    Mysql01                       226               225      4.311999  2021-03-07 03:30:00+08:00  Mysql-MySQL_3306_Innodb buffer pool pages flus...  Mysql-MySQL_3306_Innodb dblwr pages written|4....  Mysql-MySQL_3306_Innodb pages written|4.311999...
2   apache01                        81                81      3.333333  2021-03-07 03:30:00+08:00  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps...  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps...  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...
3       MG01                        92                92      2.935657  2021-03-07 03:30:00+08:00  OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNu...  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTp...
4       IG02                        90                89      2.857143  2021-03-07 03:30:00+08:00  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps...  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps...  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...
5    Redis02                       109               109      2.333333  2021-03-07 03:30:00+08:00  OSLinux-CPU_CPU_CPULoad|2.333333|2021-03-07 03...  redis-Redis_6379_Redis  (instantaneous_ops_per...  OSLinux-CPU_CPU_CPUWio|1.340295|2021-03-07 03:...
6   apache02                        82                81      2.285714  2021-03-07 03:30:00+08:00  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps...  OSLinux-NTP_197.30.1.67_NtpServerTimeOffset|1....  OSLinux-CPU_CPU_CPUWio|1.343575|2021-03-07 03:...
7   Tomcat02                       111               111      2.183020  2021-03-07 03:30:00+08:00  OSLinux-CPU_CPU_CPUCpuUtil|2.183020|2021-03-07...  OSLinux-CPU_CPU-0_SingleCpuUtil|1.915810|2021-...  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...
8    Redis01                       113               110      2.151626  2021-03-07 03:30:00+08:00  OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSe...  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWri...  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps...
9       IG01                        90                88      2.015779  2021-03-07 03:30:00+08:00  OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSe...  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps...  OSLinux-CPU_CPU_CPUWio|1.630350|2021-03-07 03:...
10  Tomcat03                       109               109      2.000000  2021-03-07 03:30:00+08:00  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTp...  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps...
11  Tomcat04                       109               109      1.818182  2021-03-07 03:30:00+08:00  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTp...  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...
12      MG02                        92                92      1.818182  2021-03-07 03:30:00+08:00  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps...  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps...  OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUti...
13  Tomcat01                       108               108      1.666667  2021-03-07 03:30:00+08:00  OSLinux-CPU_CPU-1_SingleCpuidle|1.666667|2021-...  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps...  OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUti..., 14, 14)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Ensure timezone var
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Incident window (UTC+8)
window_start = pd.Timestamp('2021-03-07 03:30:00', tz=tz)
window_end   = pd.Timestamp('2021-03-07 04:00:00', tz=tz)

# Reuse cached dataframes: metric_df, metric_global, metric_with_p95, final_df, fault_kpis
# If metric_with_p95 or fault_kpis not present, recreate needed merges
if 'metric_with_p95' not in globals():
    # merge metric_df with metric_global to get p95 per row
    metric_with_p95 = metric_df.merge(metric_global, on=['cmdb_id','kpi_name'], how='left')
# ensure ts_sh present
if 'ts_sh' not in metric_with_p95.columns:
    metric_with_p95['ts_sh'] = pd.to_datetime(metric_with_p95['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure fault_kpis exists (per-kpi rows that had anomaly_count>0)
if 'fault_kpis' not in globals():
    # final_df was computed previously with per-kpi anomaly_count
    fault_kpis = final_df[final_df['anomaly_count'] > 0].copy()

# 1) Identify top 3 Mysql02 KPI groups by severity
mysql02_faults = fault_kpis[fault_kpis['cmdb_id']=='Mysql02'].copy()
top3_mysql02 = mysql02_faults.sort_values('severity', ascending=False).head(3).reset_index(drop=True)
top3_kpis = top3_mysql02['kpi_name'].tolist()

# Prepare containers for results
anomaly_rows_list = []
kpi_summaries = []

for kpi in top3_kpis:
    # Global p95 for this KPI
    p95_row = metric_global[(metric_global['cmdb_id']=='Mysql02') & (metric_global['kpi_name']==kpi)]
    p95_value = float(p95_row['p95_value'].iloc[0]) if not p95_row.empty else np.nan

    # Rows in the incident window for this KPI
    rows_window = metric_with_p95[
        (metric_with_p95['cmdb_id']=='Mysql02') &
        (metric_with_p95['kpi_name']==kpi) &
        (metric_with_p95['ts_sh'] >= window_start) &
        (metric_with_p95['ts_sh'] <= window_end)
    ].copy()

    # max_value_in_window (over all window rows)
    max_value_in_window = rows_window['value'].max() if not rows_window.empty else np.nan

    # Filter anomaly rows where value >= global P95 (global P95 computed from full series)
    # Note: per rule, thresholds computed globally before filtering (we used metric_global)
    anomaly_rows = rows_window[rows_window['value'] >= p95_value].copy()
    # keep timestamp as tz-aware ts_sh for display
    anomaly_rows = anomaly_rows.sort_values('ts_sh').head(5)
    anomaly_rows_display = anomaly_rows[['ts_sh','cmdb_id','kpi_name','value']].rename(columns={'ts_sh':'timestamp'})

    # earliest anomaly timestamp (first row from anomaly_rows)
    if not anomaly_rows_display.empty:
        earliest_anomaly_ts = anomaly_rows_display['timestamp'].iloc[0]
    else:
        earliest_anomaly_ts = pd.NaT

    # strong_breach flag
    strong_breach = False
    if pd.isna(p95_value):
        strong_breach = False
    else:
        if p95_value == 0:
            strong_breach = (not pd.isna(max_value_in_window)) and (max_value_in_window > 0)
        else:
            strong_breach = (not pd.isna(max_value_in_window)) and ((max_value_in_window / p95_value) > 1.5)

    # Append results
    anomaly_rows_list.append(anomaly_rows_display)
    kpi_summaries.append({
        'cmdb_id':'Mysql02',
        'kpi_name': kpi,
        'p95_value': p95_value,
        'max_value_in_window': max_value_in_window,
        'earliest_anomaly_timestamp': earliest_anomaly_ts,
        'strong_breach': strong_breach
    })

# Concatenate anomaly rows (up to 15 rows: 5 per KPI)
anomalies_top3_concat = pd.concat(anomaly_rows_list, ignore_index=True).head(15)

# Prepare KPI summary DataFrame (3 rows)
kpi_summary_df = pd.DataFrame(kpi_summaries)
# Format numeric columns for compactness
kpi_summary_df['p95_value'] = kpi_summary_df['p95_value'].astype(float).round(6)
kpi_summary_df['max_value_in_window'] = kpi_summary_df['max_value_in_window'].astype(float).round(6)
kpi_summary_df['earliest_anomaly_timestamp'] = pd.to_datetime(kpi_summary_df['earliest_anomaly_timestamp'], utc=True).dt.tz_convert(tz)

# Cache for follow-ups
mysql02_top3_anomalies = anomalies_top3_concat.copy()
mysql02_top3_summary = kpi_summary_df.copy()

# Display compact outputs (up to 15 rows + 3-row summary)
mysql02_top3_anomalies, mysql02_top3_summary

```
Out[5]:
```
```python
# Compose a plain-English summary using cached results: mysql02_top3_summary, mysql02_top3_anomalies
from datetime import datetime

# Use existing cached DataFrames
sdf = mysql02_top3_summary
arows = mysql02_top3_anomalies

lines = []
lines.append("Summary of top 3 Mysql02 KPI anomalies (incident window 2021-03-07 03:30–04:00 UTC+8):")
lines.append("")

for _, r in sdf.iterrows():
    kpi = r['kpi_name']
    p95 = r['p95_value']
    maxv = r['max_value_in_window']
    ets = r['earliest_anomaly_timestamp']
    strong = r['strong_breach']
    # Gather up to 3 example anomaly values/timestamps for this KPI from the cached anomaly rows
    examples = arows[arows['kpi_name']==kpi].copy()
    ex_list = []
    for _, e in examples.head(3).iterrows():
        ts = e['timestamp']
        val = e['value']
        ex_list.append(f"{ts} -> {val}")
    ex_text = "; ".join(ex_list) if ex_list else "no anomaly rows captured"
    lines.append(f"- KPI: {kpi}")
    lines.append(f"  • global P95 = {p95:.6f}, max in window = {maxv:.6f}")
    lines.append(f"  • earliest anomaly = {ets}")
    lines.append(f"  • strong_breach (>50% above P95) = {bool(strong)}")
    lines.append(f"  • example anomaly rows (up to 3): {ex_text}")
    lines.append("")

lines.append("Overall conclusion:")
lines.append("- All three top Mysql02 KPI groups significantly exceeded their global P95 thresholds during the window.")
lines.append("- Earliest anomalies for these KPIs occurred at 2021-03-07 03:43:00+08:00.")
lines.append("- Severity is high (e.g., Opened Tables and Table open cache misses rose from P95≈0.0067 to 0.1333; Rows Read rose from P95≈5.0142 to 19.2167).")
lines.append("- These are strong breaches and warrant immediate investigation of MySQL table/cache behavior and queries around that time.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  timestamp  cmdb_id                                  kpi_name    value
0 2021-03-07 03:43:00+08:00  Mysql02            Mysql-MySQL_3306_Opened Tables   0.1333
1 2021-03-07 03:43:00+08:00  Mysql02  Mysql-MySQL_3306_Table open cache misses   0.1333
2 2021-03-07 03:43:00+08:00  Mysql02                Mysql-MySQL_3306_Rows Read  19.2167,    cmdb_id                                  kpi_name  p95_value  max_value_in_window earliest_anomaly_timestamp  strong_breach
0  Mysql02            Mysql-MySQL_3306_Opened Tables   0.006665               0.1333  2021-03-07 03:43:00+08:00           True
1  Mysql02  Mysql-MySQL_3306_Table open cache misses   0.006665               0.1333  2021-03-07 03:43:00+08:00           True
2  Mysql02                Mysql-MySQL_3306_Rows Read   5.014200              19.2167  2021-03-07 03:43:00+08:00           True)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Ensure timezone
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Use cached final_df and metric_global from previous steps
# Fallback checks (if missing, raise informative error)
if 'final_df' not in globals():
    raise RuntimeError("Expected 'final_df' in kernel (per-(cmdb_id,kpi_name) summary).")

if 'metric_global' not in globals():
    raise RuntimeError("Expected 'metric_global' in kernel (global p95 per KPI).")

# 1) Filter to Mysql02 groups
mysql02_all = final_df[final_df['cmdb_id'] == 'Mysql02'].copy()

# Count of total Mysql02 KPI groups scanned (from global thresholds)
total_mysql02_groups = int(metric_global[metric_global['cmdb_id'] == 'Mysql02'].shape[0])

# 2) Keep only groups with anomaly_count > 0
mysql02_faulty_kpis = mysql02_all[mysql02_all['anomaly_count'] > 0].copy()

# Normalize earliest_anomaly_timestamp to timezone-aware Asia/Shanghai
mysql02_faulty_kpis['earliest_anomaly_timestamp'] = pd.to_datetime(
    mysql02_faulty_kpis['earliest_anomaly_timestamp'], errors='coerce', utc=True
).dt.tz_convert(tz)

# 3) Compute severity = max_value_in_window / p95_value safely
def safe_severity(row):
    p95 = row.get('p95_value', np.nan)
    maxv = row.get('max_value_in_window', np.nan)
    if pd.isna(p95) or p95 == 0:
        if not pd.isna(maxv) and maxv > 0:
            return np.inf
        return 0.0
    if pd.isna(maxv):
        return 0.0
    return float(maxv) / float(p95)

mysql02_faulty_kpis['severity'] = mysql02_faulty_kpis.apply(safe_severity, axis=1)

# 4) Prepare compact table sorted by earliest_anomaly_timestamp ascending, limit 20
mysql02_compact = mysql02_faulty_kpis.sort_values('earliest_anomaly_timestamp', ascending=True)[
    ['cmdb_id','kpi_name','p95_value','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','severity']
].copy()

# Round numeric columns for compact display and replace inf with a large indicator
mysql02_compact['p95_value'] = mysql02_compact['p95_value'].astype(float).round(6)
mysql02_compact['max_value_in_window'] = mysql02_compact['max_value_in_window'].astype(float).round(6)
mysql02_compact['severity'] = mysql02_compact['severity'].replace([np.inf, -np.inf], np.nan).round(6)

mysql02_compact = mysql02_compact.head(20).reset_index(drop=True)

# 5) Counts
mysql02_faulty_count = int(mysql02_faulty_kpis.shape[0])

# Keep full filtered DataFrame cached as requested
# (mysql02_faulty_kpis is cached in kernel)

# Display compact results (as variables)
mysql02_compact, total_mysql02_groups, mysql02_faulty_count

```
Out[6]:
```
Summary of the Mysql02 KPI scan (filtered to cmdb_id == "Mysql02" and anomaly_count > 0):

- Counts:
  - Total Mysql02 KPI groups scanned: 226
  - Mysql02 KPI groups with anomalies in the window: 226 (every scanned KPI had anomaly_count > 0)

- Time range / timing:
  - Earliest anomalies for the returned rows occur at 2021-03-07 03:30:00+08:00 (many KPIs show that timestamp as the first anomaly).

- Observations from the compact table (first 20 rows sorted by earliest anomaly time):
  - Many KPI groups show p95_value = 0 and max_value_in_window = 0 (these appear with anomaly_count > 0 in the results).
  - Several KPI groups have nonzero p95 and nonzero max_value_in_window with severity = max / p95 = 1.0, e.g.:
    - Mysql-MySQL_3306_Qcache Free Memory (p95=1,031,832; max=1,031,832; severity=1.0)
    - Mysql-MySQL_3306_Open Files (p95=207; max=207; severity=1.0)
    - Mysql-MySQL_3306_MaxConnections (p95=2000; max=2000; severity=1.0)
    - Mysql-MySQL_3306_Max Used Connections (p95=37; max=37; severity=1.0)
    - Innodb row lock time max (p95=9522; max=9522; severity=1.0) and avg (p95=11; max=11; severity=1.0)
  - The displayed rows show many KPI groups flagged as anomalous at the very start of the incident window.

- Key takeaway:
  - All 226 monitored Mysql02 KPI groups exceeded their global P95-based anomaly condition during the incident window; many anomalies are present right at 03:30+08:00. Several MySQL-specific KPIs (open files, connections, Innodb lock times, qcache metrics) are among the affected metrics and should be prioritized for investigation.

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                        kpi_name  p95_value  anomaly_count earliest_anomaly_timestamp  max_value_in_window  severity
0   Mysql02                Mysql-MySQL_3306_Aborted Clients        0.0             20  2021-03-07 03:30:00+08:00                  0.0       0.0
1   Mysql02           Mysql-MySQL_3306_Qcache Lowmem Prunes        0.0             20  2021-03-07 03:30:00+08:00                  0.0       0.0
2   Mysql02                 Mysql-MySQL_3306_Qcache Inserts        0.0             20  2021-03-07 03:30:00+08:00                  0.0       0.0
3   Mysql02                    Mysql-MySQL_3306_Qcache Hits        0.0             20  2021-03-07 03:30:00+08:00                  0.0       0.0
4   Mysql02             Mysql-MySQL_3306_Qcache Free Memory  1031832.0             20  2021-03-07 03:30:00+08:00            1031832.0       1.0
5   Mysql02             Mysql-MySQL_3306_Qcache Free Blocks        1.0             20  2021-03-07 03:30:00+08:00                  1.0       1.0
6   Mysql02       Mysql-MySQL_3306_Opened table definitions        0.0             20  2021-03-07 03:30:00+08:00                  0.0       0.0
7   Mysql02                     Mysql-MySQL_3306_Open Files      207.0             20  2021-03-07 03:30:00+08:00                207.0       1.0
8   Mysql02             Mysql-MySQL_3306_MaxTrxRowsModified        0.0             20  2021-03-07 03:30:00+08:00                  0.0       0.0
9   Mysql02                 Mysql-MySQL_3306_MaxConnections     2000.0             20  2021-03-07 03:30:00+08:00               2000.0       1.0
10  Mysql02            Mysql-MySQL_3306_Max trx rows locked        0.0             20  2021-03-07 03:30:00+08:00                  0.0       0.0
11  Mysql02           Mysql-MySQL_3306_Max Used Connections       37.0             20  2021-03-07 03:30:00+08:00                 37.0       1.0
12  Mysql02        Mysql-MySQL_3306_Qcache Queries In Cache        0.0             20  2021-03-07 03:30:00+08:00                  0.0       0.0
13  Mysql02           Mysql-MySQL_3306_LongestTrxActiveTime        0.0             20  2021-03-07 03:30:00+08:00                  0.0       0.0
14  Mysql02             Mysql-MySQL_3306_Key Write Requests        0.0             20  2021-03-07 03:30:00+08:00                  0.0       0.0
15  Mysql02                      Mysql-MySQL_3306_Key Reads        0.0             20  2021-03-07 03:30:00+08:00                  0.0       0.0
16  Mysql02              Mysql-MySQL_3306_Key Read Requests        0.0             20  2021-03-07 03:30:00+08:00                  0.0       0.0
17  Mysql02       Mysql-MySQL_3306_Innodb row lock time max     9522.0             20  2021-03-07 03:30:00+08:00               9522.0       1.0
18  Mysql02       Mysql-MySQL_3306_Innodb row lock time avg       11.0             20  2021-03-07 03:30:00+08:00                 11.0       1.0
19  Mysql02  Mysql-MySQL_3306_Innodb row lock current waits        0.0             20  2021-03-07 03:30:00+08:00                  0.0       0.0, 226, 226)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse timezone and cached DataFrames where available
tz = pytz.timezone('Asia/Shanghai')

# Components of interest (restricted set)
components_interest = [
    "apache01","apache02",
    "Tomcat01","Tomcat02","Tomcat03","Tomcat04",
    "MG01","MG02","IG01","IG02",
    "Mysql01","Mysql02","Redis01","Redis02"
]

# Ensure we have the per-(cmdb_id,kpi_name) summary DataFrame
# Expected variable from prior steps: final_df (contains p95_value, earliest_anomaly_timestamp, max_value_in_window, anomaly_count)
if 'final_df' not in globals():
    raise RuntimeError("Expected 'final_df' in kernel (per-(cmdb_id,kpi_name) summary).")

# Work on a copy restricted to components of interest
df_comp = final_df[final_df['cmdb_id'].isin(components_interest)].copy()

# Ensure earliest_anomaly_timestamp is timezone-aware Asia/Shanghai
df_comp['earliest_anomaly_timestamp'] = pd.to_datetime(
    df_comp['earliest_anomaly_timestamp'], errors='coerce', utc=True
).dt.tz_convert(tz)

# Compute severity if not present or recompute to be safe: severity = max_value_in_window / p95_value (safe handling)
def safe_severity(row):
    p95 = row.get('p95_value', np.nan)
    maxv = row.get('max_value_in_window', np.nan)
    try:
        if pd.isna(p95) or float(p95) == 0.0:
            if not pd.isna(maxv) and float(maxv) > 0:
                return np.inf
            return 0.0
        if pd.isna(maxv):
            return 0.0
        return float(maxv) / float(p95)
    except Exception:
        return np.nan

df_comp['severity'] = df_comp.apply(safe_severity, axis=1)

# 1) Filter KPI groups to those with severity > 1.5 (strong breaches)
strong_kpis = df_comp[df_comp['severity'] > 1.5].copy()

# Cache strong_kpis in kernel for follow-up
strong_kpi_rows = strong_kpis.copy()

# 2) For each component compute aggregations
comp_agg = strong_kpis.groupby('cmdb_id', dropna=False).agg(
    strong_kpi_count = ('kpi_name','count'),
    earliest_strong_anomaly = ('earliest_anomaly_timestamp','min'),
    component_max_severity = ('severity','max')
).reset_index()

# Ensure timezone for earliest_strong_anomaly
comp_agg['earliest_strong_anomaly'] = pd.to_datetime(comp_agg['earliest_strong_anomaly'], errors='coerce', utc=True).dt.tz_convert(tz)

# 3) Prepare compact table sorted by earliest_strong_anomaly ascending and limit 20
comp_result_compact = comp_agg.sort_values('earliest_strong_anomaly', ascending=True).copy()

# Round component_max_severity for compact display and replace inf with large indicator (keep numeric)
comp_result_compact['component_max_severity'] = comp_result_compact['component_max_severity'].replace([np.inf, -np.inf], np.nan).round(6)

comp_result_compact = comp_result_compact[['cmdb_id','strong_kpi_count','earliest_strong_anomaly','component_max_severity']].head(20).reset_index(drop=True)

# 4) Two counts
scanned_components_count = int(df_comp['cmdb_id'].nunique())  # number of distinct components present in the restricted set
components_with_strong_count = int(comp_agg.shape[0])

# Cache per-component results
components_strong_summary = comp_agg.copy()

# Final compact outputs (as required: compact table and two counts)
comp_result_compact, scanned_components_count, components_with_strong_count

```
Out[7]:
```
Summary of strong breaches (severity > 1.5) within the restricted component set:

- Components scanned (restricted set): 14. Components with at least one strong KPI breach: 14 (every component in the set had at least one strong breach).
- Earliest strong breaches began shortly after the window start; top components by earliest strong anomaly:
  - IG01: 4 strong KPIs, earliest strong anomaly at 2021-03-07 03:32:00+08, component_max_severity ≈ 2.02.
  - Mysql02: 41 strong KPIs, earliest strong anomaly at 2021-03-07 03:33:00+08, component_max_severity = 20.00 (most severe).
  - Tomcat02: 10 strong KPIs, earliest strong anomaly at 2021-03-07 03:33:00+08, component_max_severity ≈ 2.18.
  - MG02, apache01, Redis02, IG02, Tomcat01, Mysql01, MG01, Tomcat04, Tomcat03, apache02, Redis01 also show strong breaches (see table for counts and times).
- Interpretation and prioritization:
  - Mysql02 is the highest-priority target (very large number of strong KPI breaches and extreme severity up to 20× the global P95).
  - IG01 and Tomcat02 show the earliest strong anomalies (around 03:32–03:33+08:00) and should be investigated early for potential causal symptoms.
  - Several other components (Mysql01, MG01, IG02, Tomcat03/04, apache01/02, Redis01/02) exhibit notable strong breaches and should be included in the incident investigation.
- Actionable next steps: focus immediate investigation on Mysql02 (highest severity and many KPIs), and examine IG01 and Tomcat02 for early signals that may indicate propagation or upstream causes.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id  strong_kpi_count   earliest_strong_anomaly  component_max_severity
0       IG01                 4 2021-03-07 03:32:00+08:00                2.015779
1    Mysql02                41 2021-03-07 03:33:00+08:00               20.000000
2   Tomcat02                10 2021-03-07 03:33:00+08:00                2.183020
3       MG02                 6 2021-03-07 03:34:00+08:00                1.818182
4   apache01                 3 2021-03-07 03:34:00+08:00                3.333333
5    Redis02                 1 2021-03-07 03:35:00+08:00                2.333333
6       IG02                 4 2021-03-07 03:38:00+08:00                2.857143
7   Tomcat01                 7 2021-03-07 03:38:00+08:00                1.666667
8    Mysql01                14 2021-03-07 03:40:00+08:00                4.311999
9       MG01                 6 2021-03-07 03:41:00+08:00                2.935657
10  Tomcat04                 4 2021-03-07 03:41:00+08:00                1.818182
11  Tomcat03                 6 2021-03-07 03:42:00+08:00                2.000000
12  apache02                 1 2021-03-07 03:46:00+08:00                2.285714
13   Redis01                 1 2021-03-07 03:58:00+08:00                2.151626, 14, 14)```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# Reuse cached trace DataFrames if available
tz = pytz.timezone('Asia/Shanghai')

# Ensure trace_global exists
if 'trace_global' not in globals():
    trace_global = (
        trace_df
        .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
        .agg(total_count='count',
             p95_value=lambda x: x.quantile(0.95),
             p5_value = lambda x: x.quantile(0.05))
        .reset_index()
    )

# Ensure trace_with_thresholds and trace_window exist
if 'trace_with_thresholds' not in globals():
    trace_with_thresholds = trace_df.merge(trace_global, on=['cmdb_id','trace_name'], how='left')
    trace_with_thresholds['ts_sh'] = pd.to_datetime(trace_with_thresholds['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

if 'trace_window' not in globals():
    window_start = pd.Timestamp('2021-03-07 03:30:00', tz=tz)
    window_end   = pd.Timestamp('2021-03-07 04:00:00', tz=tz)
    trace_window = trace_with_thresholds[(trace_with_thresholds['ts_sh'] >= window_start) & (trace_with_thresholds['ts_sh'] <= window_end)].copy()

# Total trace groups scanned
total_trace_groups_scanned = int(trace_global.shape[0])

# Find matching groups referencing Mysql02
mask = (
    (trace_global['cmdb_id'] == 'Mysql02') |
    trace_global['trace_name'].astype(str).str.contains('Mysql02|to_Mysql02|from_Mysql02', regex=True)
)
matching_groups_df = trace_global[mask].copy()
matching_groups_count = int(matching_groups_df.shape[0])

# If no matching groups, return empty results
if matching_groups_count == 0:
    matching_compact = pd.DataFrame(columns=[
        'cmdb_id','trace_name','total_count','p95_value','p5_value',
        'window_count','anomaly_count','earliest_anomaly_timestamp',
        'max_value_in_window','min_value_in_window'
    ])
    # Cache intermediates
    trace_matching_groups = matching_groups_df.copy()
    trace_matching_window_stats = matching_compact.copy()
    matching_compact, total_trace_groups_scanned, matching_groups_count
else:
    # Helper funcs
    def is_duration_trace(name):
        n = str(name).lower()
        return ('duration' in n) or ('duration_mean' in n) or ('duration_p95' in n)

    def is_rowcount_trace(name):
        return 'row_count' in str(name).lower()

    rows = []
    for row in matching_groups_df.itertuples(index=False):
        cmdb = row.cmdb_id
        tname = row.trace_name
        total_count = int(row.total_count) if not pd.isna(row.total_count) else 0
        p95 = float(row.p95_value) if not pd.isna(row.p95_value) else np.nan
        p5  = float(row.p5_value) if not pd.isna(row.p5_value) else np.nan

        # Rows in window for this group
        tw = trace_window[(trace_window['cmdb_id']==cmdb) & (trace_window['trace_name']==tname)].copy()

        window_count = int(tw.shape[0])
        max_in_window = float(tw['value'].max()) if window_count>0 and not tw['value'].isna().all() else np.nan
        min_in_window = float(tw['value'].min()) if window_count>0 and not tw['value'].isna().all() else np.nan

        # Determine anomalies per rules
        anomaly_count = 0
        earliest_anomaly_ts = pd.NaT

        if window_count > 0:
            if is_duration_trace(tname):
                anom = tw[tw['value'] >= p95]
            elif is_rowcount_trace(tname):
                anom = tw[tw['value'] <= p5]
            else:
                anom = pd.DataFrame(columns=tw.columns)

            anomaly_count = int(anom.shape[0])
            if anomaly_count > 0:
                earliest_anomaly_ts = pd.to_datetime(anom['ts_sh'].min(), utc=True).tz_convert(tz)

        rows.append({
            'cmdb_id': cmdb,
            'trace_name': tname,
            'total_count': total_count,
            'p95_value': p95,
            'p5_value': p5,
            'window_count': window_count,
            'anomaly_count': anomaly_count,
            'earliest_anomaly_timestamp': earliest_anomaly_ts,
            'max_value_in_window': max_in_window,
            'min_value_in_window': min_in_window
        })

    matching_results = pd.DataFrame(rows)

    # Compact formatting
    for col in ['p95_value','p5_value','max_value_in_window','min_value_in_window']:
        if col in matching_results.columns:
            matching_results[col] = pd.to_numeric(matching_results[col], errors='coerce').round(6)

    # Sort by earliest_anomaly_timestamp ascending (NaT last) and limit 20
    matching_compact = matching_results.sort_values('earliest_anomaly_timestamp', na_position='last').head(20).reset_index(drop=True)

    # Cache intermediates
    trace_matching_groups = matching_groups_df.copy()
    trace_matching_window_stats = matching_results.copy()

    # Final outputs
    matching_compact, total_trace_groups_scanned, matching_groups_count

```
Out[8]:
```
The search code returned no visible results (the execution result was None), so I don't have a compact table to summarize. It appears the final outputs weren't returned/printed by the last cell.

Would you like me to rerun the search now and display the compact table of trace groups referencing Mysql02 (with the requested counts and windowed anomaly stats)?

The original code execution output of IPython Kernel is also provided below for reference:

None```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz
import re

# Reuse cached trace_df if available
# Ensure timezone (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# Recompute global aggregates if not present or to be safe
if 'trace_global' not in globals():
    trace_global = (
        trace_df
        .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
        .agg(total_count='count',
             p95_value=lambda x: x.quantile(0.95),
             p5_value = lambda x: x.quantile(0.05))
        .reset_index()
    )

# Ensure trace_with_thresholds and trace_window exist and have ts_sh
if 'trace_with_thresholds' not in globals():
    trace_with_thresholds = trace_df.merge(trace_global, on=['cmdb_id','trace_name'], how='left')
    trace_with_thresholds['ts_sh'] = pd.to_datetime(trace_with_thresholds['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

if 'trace_window' not in globals():
    window_start = pd.Timestamp('2021-03-07 03:30:00', tz=tz)
    window_end   = pd.Timestamp('2021-03-07 04:00:00', tz=tz)
    trace_window = trace_with_thresholds[(trace_with_thresholds['ts_sh'] >= window_start) & (trace_with_thresholds['ts_sh'] <= window_end)].copy()

# Total number of trace groups scanned (global)
total_trace_groups_scanned = int(trace_global.shape[0])

# Find matching groups referencing Mysql02 by cmdb_id or trace_name patterns
pattern = re.compile(r'(Mysql02|to_Mysql02|from_Mysql02)', flags=re.IGNORECASE)
mask = (trace_global['cmdb_id'] == 'Mysql02') | trace_global['trace_name'].astype(str).str.contains(pattern)
matching_groups_df = trace_global[mask].copy()
matching_groups_count = int(matching_groups_df.shape[0])

# If none found, prepare empty compact results
if matching_groups_count == 0:
    matching_compact = pd.DataFrame(columns=[
        'cmdb_id','trace_name','total_count','p95_value','p5_value',
        'window_count','anomaly_count','earliest_anomaly_timestamp',
        'max_value_in_window','min_value_in_window'
    ])
    trace_matching_groups = matching_groups_df.copy()
    trace_matching_window_stats = matching_compact.copy()
else:
    # Helper classifiers
    def is_duration_trace(name):
        n = str(name).lower()
        return ('duration' in n) or ('duration_mean' in n) or ('duration_p95' in n)

    def is_rowcount_trace(name):
        return 'row_count' in str(name).lower()

    rows = []
    # Iterate matching groups and compute window stats & anomalies
    for r in matching_groups_df.itertuples(index=False):
        cmdb = r.cmdb_id
        tname = r.trace_name
        total_count = int(r.total_count) if not pd.isna(r.total_count) else 0
        p95 = float(r.p95_value) if not pd.isna(r.p95_value) else np.nan
        p5  = float(r.p5_value) if not pd.isna(r.p5_value) else np.nan

        tw = trace_window[(trace_window['cmdb_id']==cmdb) & (trace_window['trace_name']==tname)].copy()
        window_count = int(tw.shape[0])
        max_in_window = float(tw['value'].max()) if window_count>0 and not tw['value'].isna().all() else np.nan
        min_in_window = float(tw['value'].min()) if window_count>0 and not tw['value'].isna().all() else np.nan

        # Determine anomalies according to trace type
        anomaly_count = 0
        earliest_anomaly_ts = pd.NaT
        if window_count > 0:
            if is_duration_trace(tname):
                anom = tw[tw['value'] >= p95]
            elif is_rowcount_trace(tname):
                anom = tw[tw['value'] <= p5]
            else:
                anom = pd.DataFrame(columns=tw.columns)

            anomaly_count = int(anom.shape[0])
            if anomaly_count > 0:
                earliest_anomaly_ts = pd.to_datetime(anom['ts_sh'].min(), utc=True).tz_convert(tz)

        rows.append({
            'cmdb_id': cmdb,
            'trace_name': tname,
            'total_count': total_count,
            'p95_value': p95,
            'p5_value': p5,
            'window_count': window_count,
            'anomaly_count': anomaly_count,
            'earliest_anomaly_timestamp': earliest_anomaly_ts,
            'max_value_in_window': max_in_window,
            'min_value_in_window': min_in_window
        })

    matching_results = pd.DataFrame(rows)

    # Format numeric columns compactly
    for col in ['p95_value','p5_value','max_value_in_window','min_value_in_window']:
        matching_results[col] = pd.to_numeric(matching_results[col], errors='coerce').round(6)

    # Sort by earliest_anomaly_timestamp ascending (NaT last) and limit to 20
    matching_compact = matching_results.sort_values('earliest_anomaly_timestamp', na_position='last').head(20).reset_index(drop=True)

    # Cache intermediates
    trace_matching_groups = matching_groups_df.copy()
    trace_matching_window_stats = matching_results.copy()

# Final displayed variables (compact)
matching_compact, total_trace_groups_scanned, matching_groups_count

```
Out[9]:
```
I searched the cached trace data for any groups referencing Mysql02 (cmdb_id == "Mysql02" or trace_name containing "Mysql02", "to_Mysql02", or "from_Mysql02").

Results:
- Total trace groups scanned: 228
- Matching trace groups found: 0
- No trace (cmdb_id, trace_name) groups matched the requested patterns, so there are no per-group global or windowed stats to report.

Interpretation / next suggestions (brief):
- The trace dataset contains no direct traces labeled with "Mysql02" using those exact patterns. If you want broader coverage, consider searching for lowercase/partial matches like "mysql" or checking alternate naming conventions for the MySQL service in trace names or cmdb_id.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, trace_name, total_count, p95_value, p5_value, window_count, anomaly_count, earliest_anomaly_timestamp, max_value_in_window, min_value_in_window]
Index: [], 228, 0)```
```